# iDRAC Mocupでの検証
詳細は [iDRAC-Redfish-Scripting](https://github.com/dell/iDRAC-Redfish-Scripting) を参照

## 必要ライブラリのインストール

In [ ]:
%pip install requests redfish grequests multipart

## インポート

In [ ]:
import os
import subprocess
import requests

## Mocup Client 配置
[Mocup Client](https://github.com/dell/iDRAC-Redfish-Scripting/tree/master/iDRAC%20Redfish%20Mockup%20Clients) からダウンロードして目的のMocupのzipファイルをダウンロードして ./idrac_mockup_client/ に ./idrac_mockup_client/redfish となるように配置する

## [iDRAC-Redfish-Scripting](https://github.com/dell/iDRAC-Redfish-Scripting) をクローン

In [ ]:
repo_url = "https://github.com/dell/iDRAC-Redfish-Scripting.git"
target_dir = "iDRAC-Redfish-Scripting"
if not os.path.exists(target_dir):
    !git clone {repo_url}
else:
    %cd {target_dir}
    !git pull
    %cd ..

## [Redfish-Mockup-Server](https://github.com/DMTF/Redfish-Mockup-Server.git) をクローン

In [ ]:
repo_url = "https://github.com/DMTF/Redfish-Mockup-Server.git"
target_dir = "Redfish-Mockup-Server"
if not os.path.exists(target_dir):
    !git clone {repo_url}
else:
    %cd {target_dir}
    !git pull
    %cd ..

## Mocup Server 起動

In [ ]:
redfish_path = './idrac_mockup_client/'
server_command = f"python ./Redfish-Mockup-Server/redfishMockupServer.py -D {os.path.abspath(redfish_path)} -X"
mockup_server = subprocess.Popen(server_command, shell=True)

## Redfish 応答確認

In [ ]:
url = "http://127.0.0.1:8000/redfish/v1/Chassis"
response = requests.get(url)
response

In [ ]:
response.json()

## iDRAC データ取得

In [ ]:
target_ip = "127.0.0.1:8000"
username = "root"
password = "calvin"

In [ ]:
# ファイルを読み込み、https:// を http:// に置換して実行する
original_script = os.path.join("iDRAC-Redfish-Scripting", "Redfish Python", "GetFirmwareInventoryREDFISH.py")
fixed_script = "GetFirmwareInventory_Mock_http.py"

with open(original_script, 'r', encoding='utf-8') as f:
    content = f.read()
fixed_content = content.replace("'https://%s/redfish/v1", "'http://%s/redfish/v1")
with open(fixed_script, 'w', encoding='utf-8') as f:
    f.write(fixed_content)

!python {fixed_script} -ip {target_ip} -u {username} -p {password}

## Mocup Server 終了

In [ ]:
mockup_server.terminate()